# Study Partner Portal

This portal helps a student find the best people to study with.

**Each student gives us:**
- their name
- the subjects they want to study
- the times they are free (day, date, start hour, end hour)
- how many people they want to study with

**The portal then finds a group where everyone:**
1. shares at least one subject, and
2. is free on the same date for at least 1 hour at the same time.

**It can also:** login, search students, make a report and save everything in a file.

**How to use it:** run the cells from top to bottom. The last cells start the portal menu. Type your answers in the box that appears under the cell.

*We use: input/output, if-else, loops, functions, lists, dictionaries, sets, tuples, file handling and try-except.*

## Step 1: Where we store the students

Each student is a **dictionary**. All students are kept in a **list**.

- `subjects` is a **set** (no repeats, and it lets us find shared subjects easily)
- `slots` is a **list of tuples**. Each tuple is `(day, date, start_hour, end_hour)`. Hours use the 24-hour clock, so 4 pm is `16`.

We also keep the file names here.

In [14]:
MIN_HOURS = 1                      # a group needs at least 1 hour of shared free time
FILE_NAME = "students.txt"         # the students are saved in this file
REPORT_NAME = "report.txt"         # the report is saved in this file

students = []                      # the list that holds every student
next_id = 1                        # each student gets a number


def add_student(name, subjects, slots, partners):
    # Save one student into the students list.
    global next_id
    student = {"id": next_id, "name": name, "subjects": subjects, "slots": slots, "partners": partners}
    students.append(student)
    next_id = next_id + 1
    return student

## Step 2: Save and load the students (file handling)

The students must not disappear when the program closes, so we keep them in a text file. Each student is **one line**:

`name|subject1,subject2|partners|day,date,start,end;day,date,start,end`

- `save_students()` **writes** the lines into the file.
- `load_students()` **reads** the file when the portal starts.

We use `try` and `except` so the program does **not crash** if the file is missing, we are not allowed to use it, or a line is wrong. The `finally` part always closes the file.

In [15]:
def save_students(filename):
    # Write every student in the file, one line each. Returns True if it worked.
    file = None
    try:
        file = open(filename, "w")
        for s in students:
            slot_texts = []
            for slot in s["slots"]:
                slot_texts.append(slot[0] + "," + slot[1] + "," + str(slot[2]) + "," + str(slot[3]))
            line = s["name"] + "|" + ",".join(sorted(s["subjects"])) + "|" + str(s["partners"]) + "|" + ";".join(slot_texts)
            file.write(line + "\n")
        return True
    except FileNotFoundError:
        print("That folder does not exist. Students were NOT saved.")
        return False
    except PermissionError:
        print("Not allowed to write the file. Students were NOT saved.")
        return False
    finally:
        if file != None:
            file.close()


def load_students(filename):
    # Read the file and add every student to the list.
    file = None
    try:
        file = open(filename, "r")
        for line in file:
            if line.strip() == "":
                continue
            try:
                parts = line.strip().split("|")
                name = parts[0]
                subjects = set(parts[1].split(","))
                partners = int(parts[2])
                slots = []
                for text in parts[3].split(";"):
                    p = text.split(",")
                    slots.append((p[0], p[1], int(p[2]), int(p[3])))
                add_student(name, subjects, slots, partners)
            except (ValueError, IndexError):
                print("Skipped a bad line:", line.strip())
    except FileNotFoundError:
        print("No file called", filename, "yet. Starting with no students.")
    except PermissionError:
        print("Not allowed to read", filename)
    finally:
        if file != None:
            file.close()

### Load the students
The first time we run this there is no file, so we see the "No file" message. Then we add a few sample students and save them, which **creates** the file.

In [16]:
load_students(FILE_NAME)

if len(students) == 0:
    add_student("Ayesha", {"calculus", "physics"}, [("Monday", "16-11-2026", 16, 19), ("Tuesday", "17-11-2026", 10, 12)], 2)
    add_student("Bilal", {"calculus", "chemistry"}, [("Monday", "16-11-2026", 17, 20)], 2)
    add_student("Hira", {"physics", "calculus", "english"}, [("Monday", "16-11-2026", 16, 18), ("Wednesday", "18-11-2026", 9, 11)], 3)
    add_student("Usman", {"programming", "calculus"}, [("Monday", "16-11-2026", 15, 18)], 1)
    save_students(FILE_NAME)
    print("Sample students added and saved.")

print(len(students), "students are registered.")

No file called students.txt yet. Starting with no students.
Sample students added and saved.
4 students are registered.


## Step 3: Find the free time two people have in common

We compare every slot of one person with every slot of the other (a **nested loop**).
If the date is the same, the shared time starts at the *later* start and ends at the *earlier* end.

Example: `16 to 19` and `17 to 20` share `17 to 19`.

In [17]:
def common_windows(slots_a, slots_b):
    result = []
    for a in slots_a:
        for b in slots_b:
            if a[1] == b[1]:                      # same date?
                start = max(a[2], b[2])           # later start
                end = min(a[3], b[3])             # earlier end
                if end - start >= MIN_HOURS:      # long enough?
                    result.append((a[0], a[1], start, end))
    return result

# Quick test
print(common_windows([("Monday", "16-11-2026", 16, 19)], [("Monday", "16-11-2026", 17, 20)]))

[('Monday', '16-11-2026', 17, 19)]


## Step 4: Give a match score out of 100

- **60 points** for shared subjects (all of your subjects shared = 60)
- **40 points** for shared time (3 hours or more = 40)

In [18]:
def total_hours(windows):
    hours = 0
    for w in windows:
        hours = hours + (w[3] - w[2])
    return hours


def calculate_score(my_subjects, shared_subjects, windows):
    subject_points = len(shared_subjects) / len(my_subjects) * 60

    hours = total_hours(windows)
    if hours > 3:
        hours = 3
    time_points = hours / 3 * 40

    return round(subject_points + time_points)

print(calculate_score({"calculus", "physics"}, {"calculus"}, [("Monday", "16-11-2026", 17, 19)]))

57


## Step 5: Find the best group

1. Compare the student with **every other student**. Keep only people who share a subject *and* some free time. Give each one a score.
2. Put the people in order, best score first.
3. Go through them one by one. Add a person to the group **only if the whole group still shares a subject and a free time**.
4. Stop when the group is the size the student asked for.

If there are not enough people, we return the biggest group we could make.

In [19]:
def find_group(me):
    # Step 1: give a score to every other student
    candidates = []                                   # list of (score, place in the students list)
    for i in range(len(students)):
        other = students[i]
        if other["id"] != me["id"]:
            shared = me["subjects"] & other["subjects"]            # set overlap
            windows = common_windows(me["slots"], other["slots"])
            if len(shared) > 0 and len(windows) > 0:
                score = calculate_score(me["subjects"], shared, windows)
                candidates.append((score, i))

    # Step 2: best score first
    candidates.sort(reverse=True)

    # Step 3: build the group
    group_names = []
    group_shared = me["subjects"]                     # subjects the whole group shares
    group_windows = me["slots"]                       # times the whole group is free
    for item in candidates:
        if len(group_names) == me["partners"]:
            break                                     # group is full
        other = students[item[1]]
        new_shared = group_shared & other["subjects"]
        new_windows = common_windows(group_windows, other["slots"])
        if len(new_shared) > 0 and len(new_windows) > 0:
            group_names.append(other["name"])
            group_shared = new_shared
            group_windows = new_windows

    answer = {"names": group_names, "shared": group_shared, "windows": group_windows, "score": 0}
    if len(group_names) > 0:
        answer["score"] = calculate_score(me["subjects"], group_shared, group_windows)
    return answer

## Step 6: Show the result

In [20]:
def show_result(me, answer):
    print()
    print("=" * 45)
    if len(answer["names"]) == 0:
        print("Sorry", me["name"] + ", no group found yet.")
        print("Nobody shares a subject with you and is free at the")
        print("same time. Try adding more subjects or time slots,")
        print("or check again when more students register.")
        print("=" * 45)
        return

    print(me["name"] + ", your study group is ready!")
    print("Match score:", answer["score"], "/ 100")

    if len(answer["names"]) < me["partners"]:
        print("(You asked for", me["partners"], "people but only",
              len(answer["names"]), "can match your subjects and times.)")

    print()
    print("Study with:")
    for name in answer["names"]:
        print("  -", name)
    print()
    print("Shared subjects:")
    for subject in sorted(answer["shared"]):
        print("  -", subject)
    print()
    print("Everyone is free:")
    for w in answer["windows"]:
        print("  -", w[0], w[1], "from " + str(w[2]) + ":00 to " + str(w[3]) + ":00")
    print("=" * 45)

## Step 7: Login, search and report

- `user_login(name)` looks for the student with that name. It gives back the student, or `None` if there is nobody with that name.
- `search_students(word, by)` gives back a list of students whose **name** or **subject** has the word in it.
- `generate_report()` counts the students and subjects, finds the most popular subject, shows the group of every student, and saves all of it in `report.txt`.

In [21]:
def user_login(name):
    for s in students:
        if s["name"].lower() == name.strip().lower():
            return s
    return None


def search_students(word, by):
    word = word.strip().lower()
    found = []
    for s in students:
        if by == "name":
            if word in s["name"].lower():
                found.append(s)
        else:
            for subject in s["subjects"]:
                if word in subject:
                    found.append(s)
                    break                 # do not add the same student twice
    return found


def generate_report():
    lines = []
    lines.append("STUDY PARTNER PORTAL REPORT")
    lines.append("---------------------------")
    lines.append("Total students: " + str(len(students)))

    # count how many students study each subject
    counts = {}
    for s in students:
        for subject in s["subjects"]:
            if subject in counts:
                counts[subject] = counts[subject] + 1
            else:
                counts[subject] = 1

    lines.append("")
    lines.append("Students for each subject:")
    best_subject = ""
    best_count = 0
    for subject in sorted(counts):
        lines.append("  " + subject + ": " + str(counts[subject]))
        if counts[subject] > best_count:
            best_count = counts[subject]
            best_subject = subject
    lines.append("Most popular subject: " + best_subject)

    lines.append("")
    lines.append("Study group of each student:")
    for s in students:
        answer = find_group(s)
        if len(answer["names"]) == 0:
            lines.append("  " + s["name"] + ": no group yet")
        else:
            lines.append("  " + s["name"] + ": " + ", ".join(answer["names"]) + " (score " + str(answer["score"]) + ")")

    text = "\n".join(lines)

    try:
        file = open(REPORT_NAME, "w")
        file.write(text + "\n")
        file.close()
    except PermissionError:
        print("Could not save the report file.")
    return text

## Step 8: Register a new student

These functions ask the questions with `input()` and check each answer.
`ask_number()` uses `try` and `except ValueError`, so typing letters instead of a number does not crash the program. It just asks again.

In [22]:
def ask_name():
    name = input("Your name: ").strip().replace("|", "")
    while name == "":
        name = input("Please type your name: ").strip().replace("|", "")
    return name


def ask_subjects():
    subjects = set()
    while len(subjects) == 0:
        text = input("Subjects you want to study (separate with commas): ")
        for part in text.split(","):
            part = part.strip().lower()
            if part != "":
                subjects.add(part)
        if len(subjects) == 0:
            print("Please type at least one subject.")
    return subjects


def ask_number(question, low, high):
    while True:
        try:
            number = int(input(question))
            if number < low or number > high:
                print("Please type a whole number from", low, "to", high)
            else:
                return number
        except ValueError:
            print("That is not a number. Try again.")


def ask_slots():
    slots = []
    more = "yes"
    while more == "yes":
        print()
        print("Add a free time slot:")
        day = input("  Day (for example Monday): ").strip().capitalize()
        date = input("  Date (for example 16-11-2026): ").strip()
        start = ask_number("  Start hour, 24-hour clock (0-23): ", 0, 23)
        end = ask_number("  End hour, 24-hour clock (1-24): ", 1, 24)

        if end - start < MIN_HOURS:
            print("  The end must be at least 1 hour after the start. Slot not saved.")
        elif day == "" or date == "":
            print("  The day and date cannot be empty. Slot not saved.")
        else:
            slots.append((day, date, start, end))
            print("  Slot saved.")

        if len(slots) == 0:
            more = "yes"                  # we need at least one slot
        else:
            more = input("Add another slot? (yes/no): ").strip().lower()
    return slots


def register_student():
    print()
    print("--- New student ---")
    name = ask_name()
    subjects = ask_subjects()
    partners = ask_number("How many people do you want to study with (1-8)? ", 1, 8)
    slots = ask_slots()
    student = add_student(name, subjects, slots, partners)
    if save_students(FILE_NAME):
        print("Saved in", FILE_NAME)
    return student

## Step 9: The portal menu

The menu is split into small functions, so each one is easy to read:
- `show_menu()` prints the choices.
- `login_and_find()` logs the student in and shows the group.
- `search_menu()` asks what to search and prints the result.
- `run_portal()` has the `while True` loop. It keeps asking until the user types `6` to quit.

In [23]:
def show_menu():
    print()
    print("===== STUDY PARTNER PORTAL =====")
    print("1. Register a new student")
    print("2. Login and find my study group")
    print("3. Search students")
    print("4. See all registered students")
    print("5. Make a report")
    print("6. Quit")


def show_all_students():
    print()
    for s in students:
        print(s["id"], "-", s["name"], "|", ", ".join(sorted(s["subjects"])))


def login_and_find():
    name = input("Type your name: ")
    me = user_login(name)
    if me == None:
        print("No student with that name. Please register first.")
    else:
        show_result(me, find_group(me))


def search_menu():
    by = input("Search by name or subject? (name/subject): ").strip().lower()
    if by != "name" and by != "subject":
        print("Please type name or subject.")
        return
    word = input("Type the word to search: ")
    results = search_students(word, by)
    if len(results) == 0:
        print("Nobody found.")
    for s in results:
        print("-", s["name"], "|", ", ".join(sorted(s["subjects"])))


def run_portal():
    while True:
        show_menu()
        choice = input("Choose 1 to 6: ").strip()

        if choice == "1":
            me = register_student()
            show_result(me, find_group(me))
        elif choice == "2":
            login_and_find()
        elif choice == "3":
            search_menu()
        elif choice == "4":
            show_all_students()
        elif choice == "5":
            print(generate_report())
            print("(Saved in", REPORT_NAME + ")")
        elif choice == "6":
            save_students(FILE_NAME)
            print("Goodbye!")
            break
        else:
            print("Please type a number from 1 to 6.")

## Step 10: Test runs (no typing needed)
Four small tests. The last two make a mistake **on purpose** to show that the program does not crash.

In [24]:
print("TEST 1: login")
print("  Ayesha found?", user_login("Ayesha") != None)
print("  Zainab found?", user_login("Zainab") != None)

print()
print("TEST 2: search for the subject physics")
for s in search_students("physics", "subject"):
    print("  found", s["name"])

print()
print("TEST 3: open a file that does not exist")
load_students("nothing_here.txt")

print()
print("TEST 4: save in a folder that does not exist")
print("  saved?", save_students("no_such_folder/students.txt"))

TEST 1: login
  Ayesha found? True
  Zainab found? False

TEST 2: search for the subject physics
  found Ayesha
  found Hira

TEST 3: open a file that does not exist
No file called nothing_here.txt yet. Starting with no students.

TEST 4: save in a folder that does not exist
That folder does not exist. Students were NOT saved.
  saved? False


## Step 11: Start the portal

**Run 1 - login.** First a name that is not registered is typed, then a real one. After that we search and look at all students. Run the cell and type your own answers, for example `Ayesha`.

In [25]:
run_portal()


===== STUDY PARTNER PORTAL =====
1. Register a new student
2. Login and find my study group
3. Search students
4. See all registered students
5. Make a report
6. Quit


Choose 1 to 6:  2
Type your name:  Ayesha



Ayesha, your study group is ready!
Match score: 57 / 100

Study with:
  - Hira
  - Usman

Shared subjects:
  - calculus

Everyone is free:
  - Monday 16-11-2026 from 16:00 to 18:00

===== STUDY PARTNER PORTAL =====
1. Register a new student
2. Login and find my study group
3. Search students
4. See all registered students
5. Make a report
6. Quit


Choose 1 to 6:  3
Search by name or subject? (name/subject):  subject
Type the word to search:  physics


- Ayesha | calculus, physics
- Hira | calculus, english, physics

===== STUDY PARTNER PORTAL =====
1. Register a new student
2. Login and find my study group
3. Search students
4. See all registered students
5. Make a report
6. Quit


Choose 1 to 6:  4



1 - Ayesha | calculus, physics
2 - Bilal | calculus, chemistry
3 - Hira | calculus, english, physics
4 - Usman | calculus, programming

===== STUDY PARTNER PORTAL =====
1. Register a new student
2. Login and find my study group
3. Search students
4. See all registered students
5. Make a report
6. Quit


Choose 1 to 6:  6


Goodbye!


**Run 2 - register a new student.** Some wrong answers are typed on purpose (a word instead of a number). The portal just asks again. After that the report is made.

In [26]:
run_portal()


===== STUDY PARTNER PORTAL =====
1. Register a new student
2. Login and find my study group
3. Search students
4. See all registered students
5. Make a report
6. Quit


Choose 1 to 6:  1



--- New student ---


Your name:  Rabia
Subjects you want to study (separate with commas):  economics, calculus
How many people do you want to study with (1-8)?  2



Add a free time slot:


  Day (for example Monday):  Monday
  Date (for example 16-11-2026):  16-11-2026
  Start hour, 24-hour clock (0-23):  17
  End hour, 24-hour clock (1-24):  19


... (3 outputs left out)

Choose 1 to 6:  5


STUDY PARTNER PORTAL REPORT
---------------------------
Total students: 5

Students for each subject:
  calculus: 5
  chemistry: 1
  economics: 1
  english: 1
  physics: 2
  programming: 1
Most popular subject: calculus

Study group of each student:
  Ayesha: Hira, Rabia (score 43)
  Bilal: Rabia, Ayesha (score 57)
  Hira: Ayesha, Usman, Rabia (score 33)
  Usman: Hira (score 57)
  Rabia: Bilal, Ayesha (score 57)
(Saved in report.txt)

===== STUDY PARTNER PORTAL =====
1. Register a new student
2. Login and find my study group
3. Search students
4. See all registered students
5. Make a report
6. Quit


Choose 1 to 6:  6


Goodbye!


## Step 12: Look inside the files
This cell **reads** the files the portal made, so we can see that Rabia was saved and that the report was written.

In [27]:
for name in [FILE_NAME, REPORT_NAME]:
    print("=====", name, "=====")
    try:
        f = open(name, "r")
        print(f.read())
        f.close()
    except FileNotFoundError:
        print("File not found.")

===== students.txt =====
Ayesha|calculus,physics|2|Monday,16-11-2026,16,19;Tuesday,17-11-2026,10,12
Bilal|calculus,chemistry|2|Monday,16-11-2026,17,20
Hira|calculus,english,physics|3|Monday,16-11-2026,16,18;Wednesday,18-11-2026,9,11
Usman|calculus,programming|1|Monday,16-11-2026,15,18
Rabia|calculus,economics|2|Monday,16-11-2026,17,19

===== report.txt =====
STUDY PARTNER PORTAL REPORT
---------------------------
Total students: 5

Students for each subject:
  calculus: 5
  chemistry: 1
  economics: 1
  english: 1
  physics: 2
  programming: 1
Most popular subject: calculus

Study group of each student:
  Ayesha: Hira, Rabia (score 43)
  Bilal: Rabia, Ayesha (score 57)
  Hira: Ayesha, Usman, Rabia (score 33)
  Usman: Hira (score 57)
  Rabia: Bilal, Ayesha (score 57)

